In [ ]:
%pip install uv --quiet
%uv pip install pandas numpy plotly matplotlib scipy
%uv sync

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.2 environment at: c:\Users\cbutt\OneDrive\Desktop\DataScience\Project\Modern-Store-of-Value\.venv
Checked 6 packages in 26ms


In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.stooq_processor import StooqProcessor

In [7]:
stooq_tickers = {
    "Crypto ETFs": ["BITW", "IBIT", "ETHA"], 
    "Individual Stocks": ["NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"],
    "Sector ETFs": ["XLU"],
    "Broad Market ETFs": ["SPY", "VTI"],
    "Commodity ETFs (Metals)": ["GLD", "SLV", "PPLT", "PALL"],
    "Commodity ETFs (Agriculture)": ["WEAT", "SOYB", "DBA"]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [8]:
# Create flat mapping for Category
category_map = {ticker: cat for cat, ticks in stooq_tickers.items() for ticker in ticks}

with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:
    data = processor.download(stooq_tickers, start=start_date, end=end_date)

for ticker, frame in data.items():
    data[ticker] = frame.assign(Ticker=ticker, Category=category_map.get(ticker, "Other"))

combined_data = pd.concat(data.values()).reset_index()
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

In [9]:
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:
    category_map = processor.build_category_map(stooq_tickers)
    data = processor.download(
        stooq_tickers,
        start=start_date,
        end=end_date,
    )

for ticker, frame in data.items():
    data[ticker] = frame.assign(Ticker=ticker, Category=category_map[ticker])

combined_data = pd.concat(data.values()).reset_index()

# combined_data.head()
data["AAPL"].tail()

,Open,High,Low,Close,Volume,OpenInt,Ticker,Category
Date,,,,,,,,
2025-11-30,270.158,280.380,265.32,278.85,877813393,0,AAPL,Individual Stocks
2025-12-31,278.010,288.620,266.95,271.86,924529010,0,AAPL,Individual Stocks
2026-01-31,272.255,277.840,243.42,259.48,1040017271,0,AAPL,Individual Stocks
2026-02-28,260.030,280.905,255.45,264.18,988633102,0,AAPL,Individual Stocks
2026-03-26,262.410,266.530,246.00,252.89,763091456,0,AAPL,Individual Stocks


# Functions

In [ ]:
def evaluate_crisis_performance(df_long):
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    
    # Widened windows to ensure 2+ monthly points are captured
    crisis_events = {
        "2022 Bear Market":    ("2021-12-31", "2022-10-31"),
        "2023 Banking Crisis": ("2023-02-28", "2023-05-31"),
        "2025 Tariff Shock":   ("2025-03-31", "2025-04-30"),
        "2026 Iran War":       ("2026-02-28", "2026-03-31")
    }

    results = []
    for name, (start, end) in crisis_events.items():
        window = pivot_df.loc[start:end]
        if len(window) < 2: continue
            
        for ticker in window.columns:
            series = window[ticker].dropna()
            if len(series) >= 2:
                ret = (series.iloc[-1] / series.iloc[0]) - 1
                results.append({'Crisis': name, 'Ticker': ticker, 'Return': ret * 100, 'Category': category_map[ticker]})

    res_df = pd.DataFrame(results)
    gold_rets = res_df[res_df['Ticker'] == 'GLD'].set_index('Crisis')['Return']
    res_df['Excess vs Gold (%)'] = res_df.apply(lambda x: x['Return'] - gold_rets.get(x['Crisis'], 0), axis=1)
    res_df['Success'] = np.where(res_df['Excess vs Gold (%)'] > 0, "YES", "NO")

    # Faceted bar chart matching Logan's style
    fig = px.bar(res_df, x="Ticker", y="Excess vs Gold (%)", color="Category", 
                 facet_col="Crisis", facet_col_wrap=2, template="plotly_white",
                 title="Success Metric: Excess Returns vs. Gold During Crisis Windows")
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.show()
    return res_df

In [ ]:
def calculate_resilience_recovery(df_long):
    """
    Calculates the average recovery time for every asset after a 5% drop.
    Success = Faster recovery than Gold.
    """
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    
    recovery_stats = []
    for ticker in pivot_df.columns:
        prices = pivot_df[ticker].dropna()
        rolling_max = prices.cummax()
        drawdown = (prices - rolling_max) / rolling_max
        
        # Calculate how many weeks it stays below peak
        is_underwater = drawdown < 0
        # This is a simplified proxy for 'average recovery time'
        underwater_weeks = is_underwater.sum() 
        
        recovery_stats.append({'Ticker': ticker, 'Total Weeks Underwater': underwater_weeks})
        
    res_df = pd.DataFrame(recovery_stats).sort_values('Total Weeks Underwater')
    
    fig = px.bar(res_df, x='Ticker', y='Total Weeks Underwater', 
                 title="Resilience: Total Weeks Spent Below Previous Peak",
                 template="plotly_white")
    fig.show()

In [ ]:
def discover_market_shocks(df_long, target_ticker='SPY', sigma_threshold=2):
    """
    Finds dates where the market drop was more than 2 Standard Deviations from normal.
    Use these dates to find 'Crisis Events' for your report.
    """
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    rets = pivot_df[target_ticker].pct_change()
    
    mean = rets.mean()
    std = rets.std()
    
    # A 'Shock' is a drop worse than (Mean - 2*StdDev)
    shocks = rets[rets < (mean - sigma_threshold * std)]
    
    print(f"--- Systemic Shocks detected for {target_ticker} ---")
    for date, val in shocks.items():
        print(f"Shock Date: {date.date()} | Drop: {val*100:.2f}%")
        
    return shocks

In [ ]:
evaluate_crisis_performance(combined_data)

,Crisis,Ticker,Return,Category,Excess vs Gold (%),Success
0,2022 Bear Market,AAPL,-13.286146,Individual Stocks,-2.143189,NO
1,2022 Bear Market,AMD,-58.262682,Individual Stocks,-47.119725,NO
2,2022 Bear Market,AMZN,-38.554557,Individual Stocks,-27.411599,NO
3,2022 Bear Market,DBA,0.405063,Commodity ETFs (Agriculture),11.548021,YES
4,2022 Bear Market,GLD,-11.142957,Commodity ETFs (Metals),0.000000,NO
...,...,...,...,...,...,...
80,2026 Iran War,TSLA,-7.552607,Individual Stocks,9.627754,YES
81,2026 Iran War,VTI,-5.673466,Broad Market ETFs,11.506896,YES
82,2026 Iran War,WEAT,2.436863,Commodity ETFs (Agriculture),19.617225,YES
83,2026 Iran War,WMT,-4.509574,Individual Stocks,12.670788,YES


In [ ]:
calculate_resilience_recovery(combined_data)

In [ ]:
discover_market_shocks(combined_data)

--- Systemic Shocks detected for SPY ---
Shock Date: 2022-04-30 | Drop: -8.78%
Shock Date: 2022-06-30 | Drop: -8.25%
Shock Date: 2022-09-30 | Drop: -9.24%


Date
2022-04-30   -0.087770
2022-06-30   -0.082454
2022-09-30   -0.092441
Name: SPY, dtype: float64

# New Implementations

In [1]:
CRISIS_EVENTS = {
    "2022 Bear Market":    ("2021-12-31", "2022-10-31"),
    "2023 Banking Crisis": ("2023-02-28", "2023-05-31"),
    "2025 Tariff Shock":   ("2025-03-31", "2025-04-30"),
    "2026 Iran War":       ("2026-02-28", "2026-03-31")
}

RECOVERY_BUFFER_MONTHS = 12

In [15]:
def calculate_drawdown_recovery_ratio(df_long, crisis_events=CRISIS_EVENTS, recovery_buffer_months=RECOVERY_BUFFER_MONTHS, never_recovered_cap=None):
    """
    For each crisis window and each ticker, computes:
        - Max Drawdown: worst peak-to-trough drop during the crisis
        - Time to Recover: periods from trough back to pre-crisis peak
        - Ratio: Time to Recover / abs(Max Drawdown)
            - Lower = better store of value
            - Higher = slow recovery relative to how hard it fell
    
    Parameters
    ----------
    df_long               : long-format DataFrame with columns [Date, Ticker, Close, ...]
    crisis_events         : dict of { crisis_name: (start_date, end_date) }
    recovery_buffer_months: how many months beyond crisis end to search for recovery
    never_recovered_cap   : if an asset never recovers, use this fixed number as
                            Time to Recover instead of the remaining buffer length.
                            Leave as None to use the full remaining buffer length.
    """

    # Reshape from long format (one row per date per ticker)
    # to wide format (rows = dates, columns = tickers, values = closing price)
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    pivot_df.index = pd.to_datetime(pivot_df.index)

    results = []

    for crisis_name, (start, end) in crisis_events.items():
        start_dt = pd.to_datetime(start)
        end_dt   = pd.to_datetime(end)

        # Extend the window beyond the crisis end date so assets have
        # room to recover. Without this, anything that drops near the
        # end of the crisis window would appear to never recover.
        buffer_end = end_dt + pd.DateOffset(months=recovery_buffer_months)
        # Clamp to the last available date in the dataset so we don't
        # accidentally request dates that don't exist yet
        buffer_end = min(buffer_end, pivot_df.index.max())

        # full_window   = crisis period + recovery buffer (used for recovery search)
        # crisis_window = crisis period only             (used for drawdown calculation)
        full_window   = pivot_df.loc[start_dt:buffer_end]
        crisis_window = pivot_df.loc[start_dt:end_dt]

        # Need at least 2 data points to calculate a return/drawdown
        if len(crisis_window) < 2:
            continue

        for ticker in pivot_df.columns:
            # Isolate this ticker's prices for the crisis and full window,
            # dropping NaN so missing data doesn't corrupt calculations
            crisis_series = crisis_window[ticker].dropna()
            full_series   = full_window[ticker].dropna()

            if len(crisis_series) < 2:
                continue

            # ── Max Drawdown ─────────────────────────────────────────────
            # Use the rolling peak UP TO the crisis start rather than just
            # the first price inside the window. This catches assets that
            # were already declining before your crisis start date.
            pre_crisis_peak = pivot_df[ticker].loc[:start_dt].max()
            trough_val      = crisis_series.min()
            trough_date     = crisis_series.idxmin()

            # max_drawdown is a negative decimal, e.g. -0.35 means a 35% drop.
            # Multiply by 100 later when storing/dividing to get percentage form.
            max_drawdown = (trough_val - pre_crisis_peak) / pre_crisis_peak

            # Edge case: asset didn't fall at all during this crisis.
            # Ratio is 0 by definition — no drawdown, no recovery needed.
            # Logged separately so it still appears in the output table.
            if max_drawdown == 0:
                results.append({
                    'Crisis': crisis_name,
                    'Ticker': ticker,
                    'Category': category_map.get(ticker, 'Other'),
                    'Max Drawdown (%)': 0.0,
                    'Time to Recover (periods)': 0,
                    'Never Recovered': False,
                    'Ratio': 0.0,
                })
                continue

            # ── Time to Recover ──────────────────────────────────────────
            # Slice from the trough date onwards and look for the first date
            # where price climbs back to (or above) the pre-crisis peak.
            post_trough = full_series.loc[trough_date:]
            recovered   = post_trough[post_trough >= pre_crisis_peak]

            if len(recovered) == 0:
                # Asset never crossed back above its pre-crisis peak within
                # the buffer window. Options for handling this:
                #   - never_recovered_cap: treat all non-recoverers as a fixed
                #     penalty value so they don't dominate the ratio unfairly
                #   - len(post_trough): use however many periods were available,
                #     which naturally scales with how long the buffer is
                never_recovered = True
                time_to_recover = never_recovered_cap if never_recovered_cap else len(post_trough)
            else:
                never_recovered = False
                recovery_date   = recovered.index[0]
                # Count the number of periods between the trough and recovery date.
                # This is period-count based, not calendar-day based, so it respects
                # whatever frequency your data is at (daily, weekly, monthly, etc.)
                time_to_recover = len(post_trough.loc[trough_date:recovery_date])

            # ── Ratio ────────────────────────────────────────────────────
            # Core metric: how many periods did it take to recover per 1% of drawdown?
            # abs() used because max_drawdown is negative.
            # Dividing by the percentage form (not decimal) keeps the ratio
            # in a human-readable range — e.g. 3 periods per 1% drop.
            # To change the formula, this is the only line that needs to change.
            ratio = time_to_recover / abs(max_drawdown * 100)

            results.append({
                'Crisis': crisis_name,
                'Ticker': ticker,
                'Category': category_map.get(ticker, 'Other'),
                'Max Drawdown (%)': round(max_drawdown * 100, 2),
                'Time to Recover (periods)': time_to_recover,
                'Never Recovered': never_recovered,
                'Ratio': round(ratio, 2),
            })

    return pd.DataFrame(results)

In [4]:
def plot_drawdown_recovery(res_df):
    # ── Plot 1: Scatter ───────────────────────────────────────────────────
    fig1 = px.scatter(
        res_df,
        x='Max Drawdown (%)',
        y='Time to Recover (periods)',
        color='Category',
        text='Ticker',
        facet_col='Crisis',
        facet_col_wrap=2,
        template='plotly_white',
        title='Drawdown vs Recovery Time by Crisis'
    )
    fig1.update_traces(textposition='top center')
    fig1.show()

    # ── Plot 2: Ratio bar chart ───────────────────────────────────────────
    fig2 = px.bar(
        res_df,
        x='Ticker',
        y='Ratio',
        color='Category',
        facet_col='Crisis',
        facet_col_wrap=2,
        template='plotly_white',
        title='Recovery Ratio by Asset and Crisis (Lower = Better Store of Value)'
    )
    fig2.show()

In [16]:
# ── Run ───────────────────────────────────────────────────────────────────
recovery_df = calculate_drawdown_recovery_ratio(combined_data)
plot_drawdown_recovery(recovery_df)
recovery_df.head(50)

,Crisis,Ticker,Category,Max Drawdown (%),Time to Recover (periods),Never Recovered,Ratio
0,2022 Bear Market,AAPL,Individual Stocks,-22.79,12,False,0.53
1,2022 Bear Market,AMD,Individual Stocks,-62.08,13,True,0.21
2,2022 Bear Market,AMZN,Individual Stocks,-41.58,13,True,0.31
3,2022 Bear Market,DBA,Commodity ETFs (Agriculture),0.00,0,False,0.00
4,2022 Bear Market,GLD,Commodity ETFs (Metals),-14.84,4,False,0.27
5,2022 Bear Market,HD,Individual Stocks,-33.09,17,True,0.51
6,2022 Bear Market,JNJ,Individual Stocks,-4.35,3,False,0.69
7,2022 Bear Market,LOW,Individual Stocks,-31.93,17,True,0.53
8,2022 Bear Market,MSFT,Individual Stocks,-30.53,9,False,0.29
9,2022 Bear Market,NVDA,Individual Stocks,-62.82,9,False,0.14


In [17]:
# Check what prices MSFT and NVDA actually have in that window
pivot_df = combined_data.pivot(index='Date', columns='Ticker', values='Close')
pivot_df.index = pd.to_datetime(pivot_df.index)
print(pivot_df.loc["2023-02-28":"2023-05-31", ["MSFT", "NVDA", "GLD"]])

Ticker         MSFT     NVDA     GLD
Date                                
2023-02-28  246.009  23.1992  169.78
2023-03-31  284.355  27.7615  183.22
2023-04-30  303.055  27.7335  184.80
2023-05-31  324.604  37.8130  182.32


In [18]:
def summarize_and_normalize(res_df):
    """
    1. Aggregates the ratio to a single score per ticker
       (mean ratio across all crises, excluding never-recovered assets
       from the average so they don't unfairly drag scores)
    2. MinMax normalizes to a 1-10 scale where:
         1  = worst store of value (highest ratio)
         10 = best store of value  (lowest ratio)
    """

    # ── Step 1: Single metric per ticker ─────────────────────────────────
    # Exclude never-recovered rows from the mean — they carry a capped/arbitrary
    # time value that would distort the average.
    # To include them anyway, remove the filter line below.
    clean = res_df[res_df['Never Recovered'] == False].copy()

    summary = (clean
               .groupby(['Ticker', 'Category'])['Ratio']
               .agg(
                   Mean_Ratio='mean',
                   Median_Ratio='median',
                   Crisis_Count='count'        # how many crises contributed
               )
               .reset_index())

    # Flag tickers that had at least one never-recovered crisis
    never_recovered_tickers = (res_df[res_df['Never Recovered'] == True]['Ticker'].unique())
    summary['Had_Never_Recovered'] = summary['Ticker'].isin(never_recovered_tickers)

    # ── Step 2: MinMax normalize to 1–10 ─────────────────────────────────
    # Lower ratio = better, so we invert the scale:
    #   raw min (best ratio)  → score 10
    #   raw max (worst ratio) → score 1
    min_r = summary['Mean_Ratio'].min()
    max_r = summary['Mean_Ratio'].max()

    summary['Score_1_10'] = summary['Mean_Ratio'].apply(
        lambda x: 1 + 9 * (max_r - x) / (max_r - min_r)  # inverted so lower ratio = higher score
    ).round(2)

    summary = summary.sort_values('Score_1_10', ascending=False).reset_index(drop=True)

    # ── Step 3: Visualize ─────────────────────────────────────────────────
    fig = px.bar(
        summary,
        x='Ticker',
        y='Score_1_10',
        color='Category',
        text='Score_1_10',
        template='plotly_white',
        title='Store of Value Score (1–10) — Based on Recovery Ratio Across All Crises',
        labels={'Score_1_10': 'Score (10 = Best)'}
    )
    fig.update_traces(textposition='outside')
    fig.add_hline(y=5, line_dash='dash', line_color='gray',
                  annotation_text='Midpoint', annotation_position='top right')
    fig.show()

    return summary


summary_df = summarize_and_normalize(recovery_df)
summary_df

,Ticker,Category,Mean_Ratio,Median_Ratio,Crisis_Count,Had_Never_Recovered,Score_1_10
0,ETHA,Crypto ETFs,0.080000,0.080,1,True,10.00
1,PPLT,Commodity ETFs (Metals),0.140000,0.140,1,True,9.35
2,NVDA,Individual Stocks,0.156667,0.140,3,True,9.18
3,IBIT,Crypto ETFs,0.160000,0.160,1,True,9.14
4,TSLA,Individual Stocks,0.190000,0.190,1,True,8.82
5,AMD,Individual Stocks,0.190000,0.190,2,True,8.82
6,GLD,Commodity ETFs (Metals),0.200000,0.270,3,True,8.71
7,MSFT,Individual Stocks,0.226667,0.200,3,True,8.42
8,AMZN,Individual Stocks,0.295000,0.295,2,True,7.69
9,AAPL,Individual Stocks,0.390000,0.400,3,True,6.67
